# Stage 03: Localised (span-level) PCL detection (Part 5)

Utilises task2 dataset span text to get pure localised signals of where in paragh pcl is actually occuring, so the model can aggreagate these local signals and global context for more accurate paragraph-level predictions

steps: ......

## Imports & Dataset utilities

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys
import os
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from transformers import get_linear_schedule_with_warmup

# project root (one level above notebooks)
ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

from src.data.make_dataset import build_task1_task2_with_spans, validate_span_ranges, validate_span_text_alignment, truncation_rate_by_label
from src.training.metrics import stats_on_loader
from src.training.tokenization_utils import ensure_token_cache, TensorCacheDataset, PCLTokenDataset

RAW_TASK1 = ROOT / "data" / "raw" / "dontpatronizeme_pcl.tsv"
RAW_TASK2 = ROOT / "data" / "raw" / "dontpatronizeme_categories.tsv"
TRAIN_SPLIT = ROOT / "data" / "splits" / "train_semeval_parids-labels.csv"
DEV_SPLIT = ROOT / "data" / "splits" / "dev_semeval_parids-labels.csv"

train_df, dev_df, pcl_df, spans_df_norm = build_task1_task2_with_spans(
    raw_task1_path=RAW_TASK1,
    raw_task2_path=RAW_TASK2,
    train_split_path=TRAIN_SPLIT,
    dev_split_path=DEV_SPLIT,
)

print("train_df:", train_df.shape, "| positives:", int(train_df["label_bin"].sum()))
print("dev_df:", dev_df.shape, "| positives:", int(dev_df["label_bin"].sum()))
print("Example span_ranges:", train_df.loc[train_df["label_bin"].idxmax(), "span_ranges"] if len(train_df) else None)

train_df: (8375, 8) | positives: 794
dev_df: (2094, 8) | positives: 199
Example span_ranges: [(157, 242)]


## Deterministic Span Sampler for Span-First Training

### Overview: 4-Stage Span-First Training Pipeline

This section implements a **deterministic span sampler** that enables a span-first training strategy for PCL detection. Instead of training only on full paragraphs, we:

1. **Extract and sample spans** from paragraphs using linguistic segmentation
2. **Train a span classifier** (DeBERTa/ALBERT) to identify PCL at the span level
3. **Augment positive spans** with paraphrases for better generalization
4. **Aggregate span predictions** to make paragraph-level decisions

---

### Stage 1: Build Span Dataset (This Section)

**Goal**: Create a balanced span-level training dataset

**Positive Examples** (PCL spans):
- Use annotated Task2 spans from `spans_df_norm`
- These are human-labeled patronizing/condescending phrases

**Negative Examples** (non-PCL spans):
- Deterministically sample spans from non-PCL paragraphs (Task1 label=0)
- Match annotated span length distribution using quantile grid
- Ensure coverage with deterministic anchor positions

**Key Features**:
- ✅ **Deterministic**: No randomness, reproducible sampling
- ✅ **Distribution matching**: Samples match annotated span lengths
- ✅ **Sentence-aware**: Individual sentences/clauses captured as spans
- ✅ **Context preservation**: BOTH full sentences AND comma-split clauses included
- ✅ **Coverage**: Anchors + accumulation ensure all regions sampled
- ✅ **Quality filters**: Rejects mid-word starts, incomplete words, single-word spans
- ✅ **Word completion**: Always finishes last word (never cuts mid-word)
- ✅ **Text cleaning**: Removes HTML tags, normalizes entities and whitespace
- ✅ **High diversity**: Lower IoU threshold (0.4) and more anchors (4-15) for variety

---

### Algorithm Details

**1. Text Cleaning** (`clean_text_light`):
   - Remove HTML tags (`<h>`, `<p>`, `<div>`, etc.)
   - Remove HTML entities (`&nbsp;`, `&quot;`, etc.)
   - Normalize whitespace and punctuation spacing
   - Preserve sentence structure and character offsets

**2. Segmentation** (`segment_paragraph_into_units`):
   - Use spaCy for sentence segmentation
   - **Dual-mode segmentation**: Returns BOTH full sentences AND comma-split clauses
   - **Aggressive clause splitting**: Split on `,` `;` `—` even without surrounding spaces
   - Each unit = sentence or clause with char offsets
   - Captures short phrases like "dwindling hope" via comma splits
   - Preserves full sentences with commas for context

**3. Quantile-based Length Selection** (`compute_span_length_quantiles`):
   - Analyze annotated span distribution
   - Extract quantiles: [0.05, 0.10, 0.20, 0.35, 0.50, 0.65, 0.80, 0.90, 0.95]
   - Returns target lengths (in tokens) for sampling

**4. Deterministic Anchors** (`compute_deterministic_anchors`):
   - Space anchors across paragraph: `n_anchors = 4-15` (boosted for short paragraphs)
   - Evenly distribute using `np.linspace`
   - Ensures all regions have chance to contribute spans
   - Boost for short paragraphs: `min(n_units, min_anchors * 2)` for better coverage

**5. Span Construction** (`sample_spans_for_paragraph`):
   
   **Phase 1 - Individual units**: Add each sentence/clause as a span candidate
   - Captures distinct semantic units (single sentences, comma-separated phrases)
   - Ensures short, focused spans are included
   - Filters out single-word spans (uninformative, could cause keyword triggering)
   - Example: "dwindling hope" (split on comma) becomes a span
   
   **Phase 2 - Accumulated spans**: For each (anchor, target_length) pair:
   - Accumulate units forward from anchor until reaching target length
   - Extract text using char offsets
   - Align to word boundaries (avoid mid-word starts)
   - **Complete last word**: Extend span end to finish partial words
   - Apply quality filters (including single-word filter)
   - Example: Combines "They need our help." + "Without aid..." for longer spans
   
   **Phase 3 - Deduplication**:
   - Remove spans with **IoU > 0.4** (lower threshold for higher diversity)
   - Remove substring containment (span A fully inside span B)
   - Keep up to `max_spans_per_par=20` unique spans

**6. Quality Filters** (`is_well_formed_span`):
   - Reject leading punctuation (commas, periods, etc.)
   - Reject mid-word starts (e.g., "n ,", "ase of")
   - Reject incomplete words at start/end (e.g., "dise-")
   - **Reject single-word spans** (e.g., "Unfortunately,") - uninformative, 2+ words required
   - Require minimum length (3+ chars)
   - Require alphanumeric content

**7. Word Boundary Completion**:
   - **Start**: If span starts mid-word, back up to word start
   - **End**: If span ends mid-word, extend to word end
   - Ensures spans like "deserves better treatment" not "serves better tre"

---

### Example Output

**Input paragraph**:
> "Refugees need help. They deserve compassion. Without aid, they perish."

**Generated spans** (mix of individual + accumulated):
- "Refugees need help." (full sentence)
- "They deserve compassion." (full sentence)  
- "Without aid, they perish." (full sentence)
- "Refugees need help. They deserve compassion." (2 sentences accumulated)
- "They deserve compassion. Without aid, they perish." (2 sentences accumulated)
- (Full paragraph span if within target lengths)

**With commas** (e.g., "But despite the dwindling hope, Yemenis refuse..."):
**NEW: Preserves BOTH context and focused phrases:**
- "But despite the dwindling hope, Yemenis refuse to give up on others in need." (full sentence with commas)
- "But despite the dwindling hope," (before comma - focused phrase)
- "Yemenis refuse to give up on others in need." (after comma - focused phrase)
- ~~"Unfortunately,"~~ (filtered - single-word span)
- (Deduplication removes high IoU overlaps)

---

### Downstream Usage

**Stage 2**: Train span classifier on `span_train_df`
**Stage 3**: Augment positive spans with paraphrasing
**Stage 4**: Use `span_dev_df` for evaluation - score all spans, aggregate to paragraph level

In [5]:
# Import span sampler functions from module
import sys
import importlib
sys.path.append('../src')

# Force reload to get latest changes
import data.span_sampler
importlib.reload(data.span_sampler)

from data.span_sampler import (
    word_tokenize,
    TextUnit,
    SpanAnnotation,
    segment_paragraph_into_units,
    compute_span_length_quantiles,
    compute_deterministic_anchors,
    is_well_formed_span,
    compute_span_overlap_iou,
    sample_spans_for_paragraph,
    build_span_training_dataset,
    build_eval_span_candidates,
)

print("✓ Span sampler functions imported from src/data/span_sampler.py")
print("✓ Module reloaded - using latest code")

✓ Span sampler functions imported from src/data/span_sampler.py
✓ Module reloaded - using latest code


In [6]:
# Build the span datasets for training and evaluation (Stage 1)
print("="*80)
print("BUILDING SPAN DATASETS (Stage 1)")
print("="*80)

if 'spans_df_norm' in dir() and spans_df_norm is not None and not spans_df_norm.empty:
    # Compute target lengths from annotated spans
    target_lengths = compute_span_length_quantiles(spans_df_norm)
    median_len = int(spans_df_norm['span_text'].apply(lambda x: len(word_tokenize(x))).median())
    
    print(f"\nSpan length distribution from annotations:")
    print(f"  Quantile grid: {target_lengths}")
    print(f"  Median: {median_len} tokens")
    
    # Build training dataset (positive + negative spans)
    print("\n" + "="*80)
    span_train_df = build_span_training_dataset(
        train_df=train_df,
        spans_df=spans_df_norm,
        target_lengths=target_lengths,
        median_span_len=median_len,
        max_spans_per_par=20,
        negative_ratio=1.0,  # Balanced classes
    )
    
    # Build evaluation span candidates
    print("\n" + "="*80)
    span_dev_df = build_eval_span_candidates(
        eval_df=dev_df,
        target_lengths=target_lengths,
        median_span_len=median_len,
        max_spans_per_par=20,
    )
    
    print("\n" + "="*80)
    print("✓ Span datasets ready for Stage 2 (Span Classifier Training)")
    print("="*80)
else:
    print("ERROR: spans_df_norm not loaded. Run data loading cell (Cell 5) first.")
    span_train_df = None
    span_dev_df = None

BUILDING SPAN DATASETS (Stage 1)

Span length distribution from annotations:
  Quantile grid: [3, 4, 7, 10, 14, 18, 24, 31, 39]
  Median: 14 tokens



Positive spans: 100%|██████████| 2760/2760 [00:00<00:00, 36685.22it/s]



Generating negative span pool from ALL non-PCL paragraphs...


Negative spans: 100%|██████████| 7581/7581 [00:44<00:00, 170.33it/s] 



Randomly sampling 2,760 from 36,434 total negative spans...

✓ Span dataset built:
  Positive (PCL): 2,760
  Negative (non-PCL): 2,760
  Total: 5,520
  Class balance: {1: 0.5, 0: 0.5}

Generating eval span candidates for 2094 paragraphs...


Eval spans: 100%|██████████| 2094/2094 [00:14<00:00, 142.03it/s]


✓ Eval span candidates built:
  Total spans: 9,888
  Avg spans per paragraph: 4.7
  Paragraphs: PCL=199, non-PCL=1895

✓ Span datasets ready for Stage 2 (Span Classifier Training)


In [7]:
# Quick validation: Verify span sampler output quality
import random

print("="*80)
print("SPAN SAMPLER VALIDATION (5 random examples each)")
print("="*80)

# Sample from actual training data
pos_sample = span_train_df[span_train_df['span_bin'] == 1].sample(5, random_state=42)
neg_sample = span_train_df[span_train_df['span_bin'] == 0].sample(5, random_state=42)

print(f"\nPOSITIVE spans (Task2 annotations, n={len(span_train_df[span_train_df['span_bin'] == 1])}):\n")
for i, (_, row) in enumerate(pos_sample.iterrows(), 1):
    text = row['span_text'][:80] + "..." if len(row['span_text']) > 80 else row['span_text']
    print(f"  [{len(word_tokenize(row['span_text'])):2d} tok] {text}")

print(f"\nNEGATIVE spans (sampled from non-PCL, n={len(span_train_df[span_train_df['span_bin'] == 0])}):\n")
for i, (_, row) in enumerate(neg_sample.iterrows(), 1):
    text = row['span_text'][:80] + "..." if len(row['span_text']) > 80 else row['span_text']
    print(f"  [{len(word_tokenize(row['span_text'])):2d} tok] {text}")

print("="*80)

SPAN SAMPLER VALIDATION (5 random examples each)

POSITIVE spans (Task2 annotations, n=2760):

  [18 tok] be taught a skill such as handcrafts , and be paid a fair trade wage for their e...
  [11 tok] change that brings happiness to you and much more to others
  [ 4 tok] to receive papal blessings
  [13 tok] We need to carry this woman through this very dark and difficult time
  [24 tok] draw closer to the Lord so that he may pick us up and set us again on his pathwa...

NEGATIVE spans (sampled from non-PCL, n=2760):

  [ 5 tok] including her long-time refugee advocate.
  [ 7 tok] EOC seeks greater protection for pregnant women
  [17 tok] latter ; not only would it be more effective for healthier family outcomes , it ...
  [19 tok] ISPCC fundraising officer, Emma Hayden, said: " We are delighted with this suppo...
  [ 3 tok] To many others


In [8]:
# ---- choose backbone here ----
MODEL_CANDIDATES = {
    "deberta": "microsoft/deberta-v3-base",
    "albert": "albert-base-v2",
    "albert_large": "albert-large-v2",
}

MODEL_KEY = os.getenv("PCL_BACKBONE", "albert_large")  # set to "albert" to try ALBERT
MODEL_NAME = MODEL_CANDIDATES[MODEL_KEY]
MAX_LEN = 48  # Optimized: 95th percentile=47 tokens, covers 96% of training data, 25% faster than 64

def load_tokenizer(model_name: str):
    # Online first, then local cache fallback (for DNS/no-internet issues)
    try:
        return AutoTokenizer.from_pretrained(model_name, use_fast=True)
    except Exception as e:
        print(f"Tokenizer load failed ({type(e).__name__}: {e}). Trying local cache...")
        return AutoTokenizer.from_pretrained(model_name, use_fast=True, local_files_only=True)

tokenizer = load_tokenizer(MODEL_NAME)



In [9]:
# Device and AMP settings
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def _amp_settings(model_key: str, model_name: str):
    key = (model_key or "").lower()
    name = (model_name or "").lower()

    if ("albert" in key) or ("albert" in name):
        enabled = torch.cuda.is_available()
        return enabled, torch.float16, True  # FP16 + GradScaler

    if ("deberta" in key) or ("deberta" in name):
        enabled = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        return enabled, torch.bfloat16, False  # BF16, no GradScaler

    return False, None, False

USE_AMP, AMP_DTYPE, NEEDS_SCALER = _amp_settings(MODEL_KEY, MODEL_NAME)

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME} ({MODEL_KEY})")
print(f"Max length: {MAX_LEN}")
print(f"AMP enabled: {USE_AMP}, dtype: {AMP_DTYPE}, scaler needed: {NEEDS_SCALER}")
print(f"\nNote: Context window optimized to {MAX_LEN} tokens (covers 96% of training data)")
print(f"  95th percentile=47 tokens. 4x faster than 192, 25% faster than 64!")

Device: cuda
Model: albert-large-v2 (albert_large)
Max length: 48
AMP enabled: True, dtype: torch.float16, scaler needed: True

Note: Context window optimized to 48 tokens (covers 96% of training data)
  95th percentile=47 tokens. 4x faster than 192, 25% faster than 64!


## Stage 2: Span Classifier with Paragraph-Level Optimization

**Goal**: Train span-level classifier, but optimize for paragraph-level F1

**Key Innovation**: Each Optuna trial trains a span classifier, then evaluates it on paragraph-level predictions via span sampling + aggregation. This ensures hyperparameters are chosen based on the actual downstream task.

**Pipeline**:
1. Train span classifier on positive spans (Task2) + sampled negative spans
2. For each dev paragraph: sample K spans, score with model, aggregate to paragraph prediction
3. Optimize paragraph-level F1 (not span F1)

**Critical**: Strict train/dev split by par_id to prevent leakage

In [10]:
# Import span classifier utilities
import sys
import importlib

sys.path.append('../src')

# Force reload to get latest changes
import training.span_classifier
importlib.reload(training.span_classifier)

from training.span_classifier import (
    check_split_leakage,
    aggregate_multilabel_spans,
    build_span_training_dataset,
    SpanClassifier,
    aggregate_span_scores,
    evaluate_paragraph_predictions,
    compute_span_metrics,
)

print("✓ Span classifier utilities imported from src/training/span_classifier.py")

✓ Span classifier utilities imported from src/training/span_classifier.py


In [11]:
# CRITICAL: Check for data leakage between train/dev splits
print("="*80)
print("DATA LEAKAGE CHECK: Train/Dev Split Validation")
print("="*80)

train_spans_safe, dev_spans_safe, leakage_stats = check_split_leakage(
    train_df, dev_df, spans_df_norm
)

print(f"\nTask1 (paragraph-level) split:")
print(f"  Train par_ids: {leakage_stats['train_par_ids']}")
print(f"  Dev par_ids:   {leakage_stats['dev_par_ids']}")
print(f"  Overlap:       {leakage_stats['overlap_par_ids']} ← MUST BE 0")

print(f"\nTask2 (span-level) distribution:")
print(f"  Total spans:   {leakage_stats['total_spans']}")
print(f"  Train spans:   {leakage_stats['train_spans']} (from {leakage_stats['train_unique_pars_with_spans']} paragraphs)")
print(f"  Dev spans:     {leakage_stats['dev_spans']} (from {leakage_stats['dev_unique_pars_with_spans']} paragraphs)")

if leakage_stats['overlap_par_ids'] > 0:
    print("\n⚠️  WARNING: LEAKAGE DETECTED! Train and dev sets share paragraphs!")
else:
    print("\n✓ NO LEAKAGE: Train and dev sets are properly separated")

print("="*80)

DATA LEAKAGE CHECK: Train/Dev Split Validation

Task1 (paragraph-level) split:
  Train par_ids: 8375
  Dev par_ids:   2094
  Overlap:       0 ← MUST BE 0

Task2 (span-level) distribution:
  Total spans:   2760
  Train spans:   2142 (from 794 paragraphs)
  Dev spans:     618 (from 199 paragraphs)

✓ NO LEAKAGE: Train and dev sets are properly separated


In [12]:
# Verify tokenized lengths for actual training data (positive + negative sampled spans)
print("="*80)
print("TOKENIZED LENGTH VERIFICATION: Actual Training Data")
print("="*80)

# Tokenize positive spans (from Task2 annotations)
pos_texts = span_train_df[span_train_df['span_bin'] == 1]['span_text'].tolist()
pos_tokenized = tokenizer(pos_texts, add_special_tokens=True, truncation=False, padding=False)
pos_lengths = [len(ids) for ids in pos_tokenized['input_ids']]

# Tokenize negative spans (sampled from non-PCL paragraphs)
neg_texts = span_train_df[span_train_df['span_bin'] == 0]['span_text'].tolist()
neg_tokenized = tokenizer(neg_texts, add_special_tokens=True, truncation=False, padding=False)
neg_lengths = [len(ids) for ids in neg_tokenized['input_ids']]

# Combined statistics
all_lengths = pos_lengths + neg_lengths

print(f"\nPOSITIVE spans (n={len(pos_lengths)}):")
print(f"  Mean: {np.mean(pos_lengths):.1f}, Median: {np.median(pos_lengths):.0f}, Max: {np.max(pos_lengths)}")
print(f"  90th: {np.percentile(pos_lengths, 90):.0f}, 95th: {np.percentile(pos_lengths, 95):.0f}, 98th: {np.percentile(pos_lengths, 98):.0f}")

print(f"\nNEGATIVE spans (n={len(neg_lengths)}):")
print(f"  Mean: {np.mean(neg_lengths):.1f}, Median: {np.median(neg_lengths):.0f}, Max: {np.max(neg_lengths)}")
print(f"  90th: {np.percentile(neg_lengths, 90):.0f}, 95th: {np.percentile(neg_lengths, 95):.0f}, 98th: {np.percentile(neg_lengths, 98):.0f}")

print(f"\nCOMBINED training data (n={len(all_lengths)}):")
print(f"  Mean: {np.mean(all_lengths):.1f}, Median: {np.median(all_lengths):.0f}")
print(f"  95th: {np.percentile(all_lengths, 95):.0f}, 98th: {np.percentile(all_lengths, 98):.0f}")

# Check truncation at different MAX_LEN values
print(f"\n" + "="*80)
print("TRUNCATION ANALYSIS:")
for max_len_test in [48, 64, 80]:
    truncated = sum(1 for l in all_lengths if l > max_len_test)
    pct = 100 * truncated / len(all_lengths)
    print(f"  MAX_LEN={max_len_test}: {truncated}/{len(all_lengths)} truncated ({pct:.1f}%)")

print(f"\n✓ Current MAX_LEN={MAX_LEN} covers {100 * (1 - sum(1 for l in all_lengths if l > MAX_LEN) / len(all_lengths)):.1f}% of training data")
print("="*80)

TOKENIZED LENGTH VERIFICATION: Actual Training Data

POSITIVE spans (n=2760):
  Mean: 20.7, Median: 17, Max: 159
  90th: 39, 95th: 47, 98th: 60

NEGATIVE spans (n=2760):
  Mean: 20.2, Median: 18, Max: 65
  90th: 39, 95th: 45, 98th: 50

COMBINED training data (n=5520):
  Mean: 20.5, Median: 17
  95th: 46, 98th: 53

TRUNCATION ANALYSIS:
  MAX_LEN=48: 192/5520 truncated (3.5%)
  MAX_LEN=64: 40/5520 truncated (0.7%)
  MAX_LEN=80: 9/5520 truncated (0.2%)

✓ Current MAX_LEN=48 covers 96.5% of training data


In [13]:
# Build span training dataset (positive Task2 + sampled negatives)
print("="*80)
print("BUILDING SPAN TRAINING DATASET")
print("="*80)

# Recompute target lengths from TRAIN spans only (no leakage)
from data.span_sampler import word_tokenize

train_span_lengths = train_spans_safe['span_text'].apply(lambda x: len(word_tokenize(x)))
target_lengths_train = compute_span_length_quantiles(train_spans_safe)
median_len_train = int(train_span_lengths.median())

print(f"\nSpan length distribution (TRAIN spans only):")
print(f"  Quantile grid: {target_lengths_train}")
print(f"  Median: {median_len_train} tokens")
print(f"  Mean: {train_span_lengths.mean():.1f} tokens")

# Build balanced dataset
span_train_df = build_span_training_dataset(
    train_df=train_df,
    train_spans=train_spans_safe,
    sampler_fn=sample_spans_for_paragraph,
    target_lengths=target_lengths_train,
    median_span_len=median_len_train,
    negative_ratio=1.0,  # 1:1 pos:neg
    max_spans_per_par=20,
    include_hard_negatives=False,  # Start simple
)

print(f"\n{'='*80}")
print("SPAN DATASET SUMMARY:")
print(f"  Total spans: {len(span_train_df)}")
print(f"  Positive:    {(span_train_df['span_bin'] == 1).sum()} ({(span_train_df['span_bin'] == 1).mean():.1%})")
print(f"  Negative:    {(span_train_df['span_bin'] == 0).sum()} ({(span_train_df['span_bin'] == 0).mean():.1%})")

# Show examples
print(f"\n{'='*80}")
print("SAMPLE SPANS:")
print("\nPositive (Task2 annotations):")
for i, row in span_train_df[span_train_df['span_bin'] == 1].sample(3, random_state=42).iterrows():
    print(f"  [{len(word_tokenize(row['span_text'])):2d} tok] {row['span_text'][:100]}")

print("\nNegative (sampled from non-PCL):")
for i, row in span_train_df[span_train_df['span_bin'] == 0].sample(3, random_state=42).iterrows():
    print(f"  [{len(word_tokenize(row['span_text'])):2d} tok] {row['span_text'][:100]}")

print("="*80)

BUILDING SPAN TRAINING DATASET

Span length distribution (TRAIN spans only):
  Quantile grid: [3, 5, 7, 11, 15, 19, 25, 32, 40]
  Median: 15 tokens
  Mean: 16.9 tokens

SPAN DATASET SUMMARY:
  Total spans: 3970
  Positive:    1985 (50.0%)
  Negative:    1985 (50.0%)

SAMPLE SPANS:

Positive (Task2 annotations):
  [21 tok] People across Australia ordered pizzas to be delivered on Saturday night , with the ample leftovers 
  [25 tok] For her unwavering commitment to aiding those most in need , Mother Teresa stands out as one of the 
  [35 tok] Project Kickstart helps immigrants looking for work to find it , often through Hoffman 's incredible

Negative (sampled from non-PCL):
  [22 tok] However, this year he met a group of enthusiastic disabled runners as well Paralympic Games competit
  [32 tok] Stefanovic said immigrants " from faraway lands " had " helped make Australia " and Mr Dutton had " 
  [21 tok] There 's a certain level of exploitation that happens to a lot of workers when the

In [14]:
# Setup for span classifier training
from torch.utils.data import DataLoader, TensorDataset
import torch

# Model settings (reuse from earlier)
print(f"Model: {MODEL_NAME}")
print(f"Max length: {MAX_LEN}")
print(f"Device: {DEVICE}")
print(f"AMP: {USE_AMP}, dtype: {AMP_DTYPE}")

# Tokenize span dataset
print("\nTokenizing span dataset...")
span_encodings = tokenizer(
    span_train_df['span_text'].tolist(),
    max_length=MAX_LEN,
    padding='max_length',
    truncation=True,
    return_tensors='pt',
)

span_labels = torch.tensor(span_train_df['span_bin'].values, dtype=torch.float32)

# Create dataset
span_dataset = TensorDataset(
    span_encodings['input_ids'],
    span_encodings['attention_mask'],
    span_encodings.get('token_type_ids', torch.zeros_like(span_encodings['input_ids'])),
    span_labels,
)

print(f"✓ Span dataset ready: {len(span_dataset)} examples")

# Compute class weights for balanced training
pos_count = int(span_labels.sum())
neg_count = len(span_labels) - pos_count
pos_weight = 1.0 / max(pos_count, 1)
neg_weight = 1.0 / max(neg_count, 1)

print(f"\nClass distribution:")
print(f"  Positive: {pos_count} (weight: {pos_weight:.4f})")
print(f"  Negative: {neg_count} (weight: {neg_weight:.4f})")

Model: albert-large-v2
Max length: 48
Device: cuda
AMP: True, dtype: torch.float16

Tokenizing span dataset...
✓ Span dataset ready: 3970 examples

Class distribution:
  Positive: 1985 (weight: 0.0005)
  Negative: 1985 (weight: 0.0005)


In [15]:
# Optuna setup for span classifier + paragraph aggregation
import optuna
from optuna.pruners import MedianPruner
import random
import gc

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)

# NEW study name to avoid overwriting previous experiments
OPTUNA_DIR_SPAN = ROOT / "runs" / "optuna_stage05_span_paragraph_agg"
OPTUNA_DIR_SPAN.mkdir(parents=True, exist_ok=True)

storage_url = f"sqlite:///{(OPTUNA_DIR_SPAN / 'study.db').as_posix()}"
study = optuna.create_study(
    study_name="stage05_span_paragraph_agg",  # NEW study name
    direction="maximize",
    storage=storage_url,
    load_if_exists=True,
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=1, interval_steps=1),
)

print("="*80)
print("OPTUNA STUDY: Span Classifier + Paragraph Aggregation")
print("="*80)
print(f"Storage: {storage_url}")
print(f"Study name: stage05_span_paragraph_agg")
print(f"Trials so far: {len(study.trials)}")
print(f"Optimization target: Paragraph-level F1 on dev set")
print("="*80)

[I 2026-03-03 07:08:36,094] Using an existing study with name 'stage05_span_paragraph_agg' instead of creating a new one.


OPTUNA STUDY: Span Classifier + Paragraph Aggregation
Storage: sqlite:////home/joshua_killa/doc/y3/PCL-detection/runs/optuna_stage05_span_paragraph_agg/study.db
Study name: stage05_span_paragraph_agg
Trials so far: 2
Optimization target: Paragraph-level F1 on dev set


In [16]:
# Optuna objective: train span classifier, evaluate on paragraph aggregation
from torch.optim import AdamW
from torch.utils.data import DataLoader, WeightedRandomSampler
from transformers import get_linear_schedule_with_warmup

# Fixed hyperparameters
SPAN_EPOCHS = 7
SPAN_BATCH_SIZE = 32
EARLY_STOP_PATIENCE = 3

def objective(trial: optuna.Trial):
    set_seed(SEED)
    
    # Hyperparameters to tune
    # --- Span training ---
    lr = trial.suggest_float("lr", 1e-6, 5e-5, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    dropout = trial.suggest_float("dropout", 0.0, 0.3)
    
    # Loss weighting (upweight negatives to reduce FP)
    use_class_weights = trial.suggest_categorical("use_class_weights", [True, False])
    if use_class_weights:
        alpha_neg = trial.suggest_float("alpha_neg", 0.5, 5.0)  # Upweight negatives
    else:
        alpha_neg = 1.0
    
    # --- Paragraph aggregation ---
    num_spans_per_par = trial.suggest_int("num_spans_per_par", 5, 40)
    agg_mode = trial.suggest_categorical("agg_mode", ["max", "topk_mean"])
    
    if agg_mode == "topk_mean":
        topk = trial.suggest_int("topk", 1, 5)
    else:
        topk = 3  # Default (unused)
    
    para_thresh = trial.suggest_float("para_thresh", 0.3, 0.8)
    
    # Store metadata
    trial.set_user_attr("model_name", MODEL_NAME)
    trial.set_user_attr("model_key", MODEL_KEY)
    trial.set_user_attr("max_len", int(MAX_LEN))
    trial.set_user_attr("epochs", int(SPAN_EPOCHS))
    trial.set_user_attr("batch_size", int(SPAN_BATCH_SIZE))
    
    try:
        # Create weighted sampler for balanced batches
        sample_weights_tensor = torch.tensor([
            neg_weight * alpha_neg if label == 0 else pos_weight 
            for label in span_labels
        ], dtype=torch.double)
        
        sampler = WeightedRandomSampler(
            sample_weights_tensor,
            num_samples=len(sample_weights_tensor),
            replacement=True,
        )
        
        train_loader = DataLoader(
            span_dataset,
            batch_size=SPAN_BATCH_SIZE,
            sampler=sampler,
        )
        
        # Initialize model
        model = SpanClassifier(MODEL_NAME, dropout=dropout).to(DEVICE)
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        
        total_steps = len(train_loader) * SPAN_EPOCHS
        warmup_steps = int(0.1 * total_steps)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )
        
        # Optional weighted loss
        if use_class_weights:
            pos_weight_tensor = torch.tensor([alpha_neg], dtype=torch.float32).to(DEVICE)
        else:
            pos_weight_tensor = None
        
        local_scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))
        
        best_para_f1 = -1.0
        no_improve = 0
        best_model_state = None  # Track best model
        
        # Training loop
        for epoch in range(SPAN_EPOCHS):
            model.train()
            epoch_loss = 0.0
            
            for batch_idx, batch in enumerate(train_loader):
                input_ids, attention_mask, token_type_ids, labels = batch
                
                input_ids = input_ids.to(DEVICE)
                attention_mask = attention_mask.to(DEVICE)
                token_type_ids = token_type_ids.to(DEVICE)
                labels = labels.to(DEVICE)
                
                optimizer.zero_grad()
                
                if USE_AMP:
                    with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                        outputs = model(
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                            token_type_ids=token_type_ids,
                            labels=labels,
                        )
                        loss = outputs['loss']
                        
                        # Apply manual weighting if needed
                        if pos_weight_tensor is not None:
                            logits = outputs['logits']
                            loss = F.binary_cross_entropy_with_logits(
                                logits.float(),
                                labels.float(),
                                pos_weight=pos_weight_tensor,
                            )
                else:
                    outputs = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        token_type_ids=token_type_ids,
                        labels=labels,
                    )
                    loss = outputs['loss']
                    
                    if pos_weight_tensor is not None:
                        logits = outputs['logits']
                        loss = F.binary_cross_entropy_with_logits(
                            logits.float(),
                            labels.float(),
                            pos_weight=pos_weight_tensor,
                        )
                
                if local_scaler.is_enabled():
                    local_scaler.scale(loss).backward()
                    local_scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    local_scaler.step(optimizer)
                    local_scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                
                scheduler.step()
                epoch_loss += loss.item()
            
            avg_loss = epoch_loss / len(train_loader)
            
            # Evaluate on paragraph-level (THE OPTIMIZATION TARGET)
            para_metrics = evaluate_paragraph_predictions(
                model=model,
                paragraph_df=dev_df,
                sampler_fn=sample_spans_for_paragraph,
                target_lengths=target_lengths_train,
                median_span_len=median_len_train,
                tokenizer=tokenizer,
                max_len=MAX_LEN,
                device=DEVICE,
                use_amp=USE_AMP,
                amp_dtype=AMP_DTYPE,
                num_spans_per_par=num_spans_per_par,
                agg_mode=agg_mode,
                topk=topk,
                span_thresh=None,  # No span filtering for now
                para_thresh=para_thresh,
            )
            
            para_f1 = para_metrics['f1']
            
            # Optional: also compute span-level metrics for diagnostics
            if len(dev_spans_safe) > 0 and epoch == SPAN_EPOCHS - 1:  # Only last epoch
                span_metrics = compute_span_metrics(
                    model=model,
                    span_df=dev_spans_safe.rename(columns={
                        'span_start_norm': 'span_start_char',
                        'span_finish_norm': 'span_end_char',
                    }).assign(span_bin=1),  # All dev spans are positive
                    tokenizer=tokenizer,
                    max_len=MAX_LEN,
                    device=DEVICE,
                    use_amp=USE_AMP,
                    amp_dtype=AMP_DTYPE,
                    threshold=0.5,
                )
                trial.set_user_attr("dev_span_metrics", {k: float(v) for k, v in span_metrics.items()})
            
            print(
                f"[Trial {trial.number}] Epoch {epoch+1}/{SPAN_EPOCHS} | "
                f"loss={avg_loss:.4f} | "
                f"para_f1={para_f1:.4f} para_prec={para_metrics['precision']:.4f} para_rec={para_metrics['recall']:.4f} | "
                f"para_acc={para_metrics['accuracy']:.4f} (pos={para_metrics['pos_acc']:.4f}, neg={para_metrics['neg_acc']:.4f})"
            )
            
            # Store epoch metrics
            trial.set_user_attr(f"epoch_{epoch}_para", {k: float(v) for k, v in para_metrics.items()})
            
            # Track best
            if para_f1 > best_para_f1:
                best_para_f1 = para_f1
                trial.set_user_attr("best_epoch", int(epoch))
                trial.set_user_attr("best_para_metrics", {k: float(v) for k, v in para_metrics.items()})
                # Save best model state
                best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
            
            # Report to Optuna
            trial.report(para_f1, step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
            
            # Early stopping
            if no_improve >= EARLY_STOP_PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break
        
        # Save best model checkpoint
        if best_model_state is not None:
            checkpoint_path = OPTUNA_DIR_SPAN / f"trial_{trial.number:04d}_best.pt"
            torch.save({
                'model_state_dict': best_model_state,
                'trial_number': trial.number,
                'best_para_f1': best_para_f1,
                'hyperparams': trial.params,
            }, checkpoint_path)
            trial.set_user_attr("checkpoint_path", str(checkpoint_path))
        
        return float(best_para_f1)
    
    except torch.cuda.OutOfMemoryError:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise optuna.TrialPruned()
    
    finally:
        del model, optimizer, scheduler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("✓ Objective function defined")

✓ Objective function defined


In [17]:
# Run Optuna optimization
N_TRIALS = 50

print("="*80)
print(f"STARTING OPTUNA OPTIMIZATION: {N_TRIALS} trials")
print("="*80)

study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True)

print("\n" + "="*80)
print("OPTIMIZATION COMPLETE")
print("="*80)
print(f"\nBest paragraph-level F1: {study.best_value:.4f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# Show best trial metrics
best_trial = study.best_trial
if "best_para_metrics" in best_trial.user_attrs:
    print(f"\nBest trial paragraph metrics:")
    for key, value in best_trial.user_attrs["best_para_metrics"].items():
        print(f"  {key}: {value:.4f}")

print("="*80)

STARTING OPTUNA OPTIMIZATION: 50 trials


Loading weights: 100%|██████████| 25/25 [00:00<00:00, 500.80it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_727761/3010086110.py:82: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  local_scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))


[Trial 2] Epoch 1/7 | loss=0.6024 | para_f1=0.2307 para_prec=0.1560 para_rec=0.4422 | para_acc=0.7197 (pos=0.4422, neg=0.7488)
[Trial 2] Epoch 2/7 | loss=0.3963 | para_f1=0.2581 para_prec=0.1736 para_rec=0.5025 | para_acc=0.7254 (pos=0.5025, neg=0.7488)
[Trial 2] Epoch 3/7 | loss=0.3160 | para_f1=0.2626 para_prec=0.1818 para_rec=0.4724 | para_acc=0.7479 (pos=0.4724, neg=0.7768)
[Trial 2] Epoch 4/7 | loss=0.2662 | para_f1=0.2625 para_prec=0.1765 para_rec=0.5126 | para_acc=0.7264 (pos=0.5126, neg=0.7488)
[Trial 2] Epoch 5/7 | loss=0.2400 | para_f1=0.2605 para_prec=0.1694 para_rec=0.5628 | para_acc=0.6963 (pos=0.5628, neg=0.7103)
[Trial 2] Epoch 6/7 | loss=0.2166 | para_f1=0.2597 para_prec=0.1679 para_rec=0.5729 | para_acc=0.6896 (pos=0.5729, neg=0.7018)
  Early stopping at epoch 6


[I 2026-03-03 07:19:05,161] Trial 2 finished with value: 0.26256983240223464 and parameters: {'lr': 1.3363974872329465e-06, 'weight_decay': 0.05688899452969007, 'dropout': 0.10656076141259616, 'use_class_weights': False, 'num_spans_per_par': 22, 'agg_mode': 'max', 'para_thresh': 0.7854341248579142}. Best is trial 2 with value: 0.26256983240223464.
Loading weights: 100%|██████████| 25/25 [00:00<00:00, 338.41it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from dif

[Trial 3] Epoch 1/7 | loss=0.8090 | para_f1=0.2423 para_prec=0.2157 para_rec=0.2764 | para_acc=0.8357 (pos=0.2764, neg=0.8945)
[Trial 3] Epoch 2/7 | loss=0.4254 | para_f1=0.2471 para_prec=0.2979 para_rec=0.2111 | para_acc=0.8777 (pos=0.2111, neg=0.9478)
[Trial 3] Epoch 3/7 | loss=0.3863 | para_f1=0.2418 para_prec=0.3458 para_rec=0.1859 | para_acc=0.8892 (pos=0.1859, neg=0.9631)
[Trial 3] Epoch 4/7 | loss=0.2617 | para_f1=0.2342 para_prec=0.3162 para_rec=0.1859 | para_acc=0.8844 (pos=0.1859, neg=0.9578)
[Trial 3] Epoch 5/7 | loss=0.1646 | para_f1=0.2305 para_prec=0.3033 para_rec=0.1859 | para_acc=0.8820 (pos=0.1859, neg=0.9551)
  Early stopping at epoch 5


...

[Trial 50] Epoch 1/7 | loss=0.7722 | para_f1=0.2770 para_prec=0.1884 para_rec=0.5226 | para_acc=0.7407 (pos=0.5226, neg=0.7636)
[Trial 50] Epoch 2/7 | loss=0.3861 | para_f1=0.3016 para_prec=0.2128 para_rec=0.5176 | para_acc=0.7722 (pos=0.5176, neg=0.7989)
[Trial 50] Epoch 3/7 | loss=0.2477 | para_f1=0.2978 para_prec=0.2018 para_rec=0.5678 | para_acc=0.7455 (pos=0.5678, neg=0.7641)
[Trial 50] Epoch 4/7 | loss=0.2482 | para_f1=0.3290 para_prec=0.2423 para_rec=0.5126 | para_acc=0.8013 (pos=0.5126, neg=0.8317)
[Trial 50] Epoch 5/7 | loss=0.1836 | para_f1=0.2744 para_prec=0.1804 para_rec=0.5729 | para_acc=0.7120 (pos=0.5729, neg=0.7266)
[Trial 50] Epoch 6/7 | loss=0.1407 | para_f1=0.2658 para_prec=0.1682 para_rec=0.6332 | para_acc=0.6676 (pos=0.6332, neg=0.6712)
[Trial 50] Epoch 7/7 | loss=0.1057 | para_f1=0.2691 para_prec=0.1750 para_rec=0.5829 | para_acc=0.6991 (pos=0.5829, neg=0.7113)
  Early stopping at epoch 7


[I 2026-03-03 14:46:42,939] Trial 50 finished with value: 0.32903225806451614 and parameters: {'lr': 1.0583272759135636e-05, 'weight_decay': 0.06376190569394925, 'dropout': 0.13765499459240835, 'use_class_weights': True, 'alpha_neg': 3.9080993145598533, 'num_spans_per_par': 32, 'agg_mode': 'topk_mean', 'topk': 1, 'para_thresh': 0.7616526736789904}. Best is trial 36 with value: 0.359375.
Loading weights: 100%|██████████| 25/25 [00:00<00:00, 344.13it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECT

[Trial 51] Epoch 1/7 | loss=0.7490 | para_f1=0.2235 para_prec=0.1338 para_rec=0.6784 | para_acc=0.5521 (pos=0.6784, neg=0.5388)
[Trial 51] Epoch 2/7 | loss=0.3588 | para_f1=0.2278 para_prec=0.1336 para_rec=0.7739 | para_acc=0.5014 (pos=0.7739, neg=0.4728)


[I 2026-03-03 14:50:18,285] Trial 51 pruned. 



OPTIMIZATION COMPLETE

Best paragraph-level F1: 0.3594

Best hyperparameters:
  lr: 7.376874377190617e-06
  weight_decay: 0.07092529902853466
  dropout: 0.0944780448313792
  use_class_weights: True
  alpha_neg: 3.9350882049579523
  num_spans_per_par: 21
  agg_mode: topk_mean
  topk: 1
  para_thresh: 0.7467098910945877

Best trial paragraph metrics:
  f1: 0.3594
  precision: 0.2939
  recall: 0.4623
  accuracy: 0.8434
  pos_acc: 0.4623
  neg_acc: 0.8834


In [18]:
# Clear CUDA cache after optimization
torch.cuda.empty_cache()
print("✓ CUDA cache cleared")

✓ CUDA cache cleared


In [19]:
# Analyze Optuna study results
import pandas as pd

# Get all trials as DataFrame
trials_df = study.trials_dataframe()

# Show top 5 trials
print("="*80)
print("TOP 5 TRIALS (by paragraph F1)")
print("="*80)

top_trials = trials_df.nlargest(5, 'value')[
    ['number', 'value', 'params_lr', 'params_dropout', 'params_alpha_neg', 
     'params_num_spans_per_par', 'params_topk', 'params_para_thresh', 'state']
]

for idx, row in top_trials.iterrows():
    print(f"\nTrial {int(row['number'])}: F1={row['value']:.4f} ({row['state']})")
    print(f"  lr={row['params_lr']:.2e}, dropout={row['params_dropout']:.3f}")
    if pd.notna(row.get('params_alpha_neg')):
        print(f"  alpha_neg={row['params_alpha_neg']:.2f}")
    print(f"  spans/par={int(row['params_num_spans_per_par'])}, topk={int(row['params_topk'])}, thresh={row['params_para_thresh']:.3f}")

print("\n" + "="*80)
print("HYPERPARAMETER IMPORTANCE (Top 5)")
print("="*80)

try:
    importance = optuna.importance.get_param_importances(study)
    for i, (param, imp) in enumerate(sorted(importance.items(), key=lambda x: -x[1])[:5], 1):
        print(f"  {i}. {param}: {imp:.4f}")
except Exception as e:
    print(f"  Could not compute importance: {e}")

print("="*80)

TOP 5 TRIALS (by paragraph F1)

Trial 36: F1=0.3594 (COMPLETE)
  lr=7.38e-06, dropout=0.094
  alpha_neg=3.94
  spans/par=21, topk=1, thresh=0.747

Trial 32: F1=0.3333 (COMPLETE)
  lr=6.97e-06, dropout=0.145
  alpha_neg=3.86
  spans/par=36, topk=1, thresh=0.747

Trial 34: F1=0.3333 (COMPLETE)
  lr=7.53e-06, dropout=0.133
  alpha_neg=3.87
  spans/par=27, topk=1, thresh=0.756

Trial 50: F1=0.3290 (COMPLETE)
  lr=1.06e-05, dropout=0.138
  alpha_neg=3.91
  spans/par=32, topk=1, thresh=0.762

Trial 33: F1=0.3284 (COMPLETE)
  lr=8.10e-06, dropout=0.128
  alpha_neg=3.77
  spans/par=36, topk=1, thresh=0.760

HYPERPARAMETER IMPORTANCE (Top 5)
  1. weight_decay: 0.5340
  2. lr: 0.1760
  3. para_thresh: 0.1702
  4. dropout: 0.0687
  5. num_spans_per_par: 0.0471
